#### This file uses last 7 days of dumped data then to interpolate and fix the gaps

In [ ]:
import pandas as pd

# Load your CSV
df = pd.read_csv("last7days.csv", parse_dates=["clock", "next_clock"])

# Set clock as datetime index
df['clock'] = pd.to_datetime(df['clock'])

# Function to fill gaps for each contour_id
def interpolate_contour(group):
    # Set index
    contour_id = group['contour_id'].iloc[0]  # keep contour_id
    fuel_coef = group['fuel_coefficient'].iloc[0]  # keep fuel_coef
    group = group.set_index('clock').sort_index()
    
    # Create a complete 15-min interval index
    full_index = pd.date_range(start=group.index.min(), end=group.index.max(), freq='15T')
    
    # Reindex to include missing timestamps
    group = group.reindex(full_index)
    
    group['contour_id'] = contour_id
    group['fuel_coefficient'] = fuel_coef

    # Interpolate numeric columns
    group['energy_export'] = group['energy_export'].interpolate(method='time')
    group['energy_import'] = group['energy_import'].interpolate(method='time')
    
    # Reset index
    group = group.reset_index().rename(columns={'index': 'clock'})

    return group[['contour_id','clock','fuel_coefficient', 'energy_import', 'energy_export']]

# Apply per contour_id
result = df.groupby('contour_id').apply(interpolate_contour).reset_index(drop=True)
print(result.head())
# Optional: inspect


/tmp/ipykernel_185111/3366665295.py:17: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_index = pd.date_range(start=group.index.min(), end=group.index.max(), freq='15T')
/tmp/ipykernel_185111/3366665295.py:17: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_index = pd.date_range(start=group.index.min(), end=group.index.max(), freq='15T')
/tmp/ipykernel_185111/3366665295.py:17: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_index = pd.date_range(start=group.index.min(), end=group.index.max(), freq='15T')
/tmp/ipykernel_185111/3366665295.py:17: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_index = pd.date_range(start=group.index.min(), end=group.index.max(), freq='15T')
/tmp/ipykernel_185111/3366665295.py:17: FutureWarning: 'T' is deprecated and will be

   contour_id               clock  fuel_coefficient  energy_import  \
0    13836498 2025-06-01 12:15:00                 1      6517117.0   
1    13836498 2025-06-01 12:30:00                 1      6517160.0   
2    13836498 2025-06-01 12:45:00                 1      6517205.0   
3    13836498 2025-06-01 13:00:00                 1      6517249.0   
4    13836498 2025-06-01 13:15:00                 1      6517296.0   

   energy_export  
0          469.0  
1          469.0  
2          469.0  
3          469.0  
4          469.0  


/tmp/ipykernel_185111/3366665295.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df.groupby('contour_id').apply(interpolate_contour).reset_index(drop=True)


In [16]:
for index, row in result.iterrows():
    print(row)
    break


contour_id                  13836498
clock            2025-06-01 12:15:00
energy_import              6517117.0
energy_export                  469.0
Name: 0, dtype: object


In [24]:
def insert_contour(conn, contour_id, fuel_coef):
    with conn.cursor() as cursor:
        insert_query = """
        INSERT INTO interpolated.contour (contour_id, fuel_coefficient)
        VALUES (%s, %s)
        """
        cursor.execute(insert_query, (contour_id, fuel_coef))
    conn.commit()
def insert_row(conn, row):
    with conn.cursor() as cursor:
        insert_query = """
        INSERT INTO interpolated.contour_data (contour_id, clock, energy_import, energy_export)
        VALUES (%s, %s, %s, %s)
        """
        cursor.execute(insert_query, (row['contour_id'], row['clock'], row['energy_import'], row['energy_export']))
    conn.commit()

In [ ]:
import psycopg2

try:
    conn = psycopg2.connect(
        host="localhost",
        database="postgres",
        user="postgres",
        password="11111",
        port="5432"
    )

    # If connection is successful, you can now interact with the database
    print("Connection to PostgreSQL successful!")

except psycopg2.Error as e:
    print(f"Error connecting to PostgreSQL: {e}")


Connection to PostgreSQL successful!


In [26]:
conn.close()

In [28]:
inserted_contours = set()
for index, row in result.iterrows():
    if row['contour_id'] not in inserted_contours:
        insert_contour(conn, row['contour_id'], row['fuel_coefficient'])
        inserted_contours.add(row['contour_id'])
    insert_row(conn, row)